# StyleMatch Style Embedding Block-Length Audit

Aggregates source-heldout literary chunks into longer blocks and reruns StyleDistance recall.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/stylematch_v1')
assert (Path.cwd() / 'scripts/make_block_splits.py').exists(), 'Run from cloned style_matching repo.'
assert (Path.cwd() / 'scripts/style_embedding_recall.py').exists(), 'Run from cloned style_matching repo.'


In [ ]:
!pip -q install sentence-transformers pandas pyarrow scikit-learn


In [ ]:
base_split = ROOT / 'data/literary/meta/literary_source_heldout_splits.parquet'

for n in [2, 3, 4]:
    block_split = ROOT / f'data/literary/meta/literary_source_heldout_blocks_{n}.parquet'
    block_report = ROOT / f'data/literary/meta/literary_source_heldout_blocks_{n}_report.json'
    !python scripts/make_block_splits.py --input "{base_split}" --output "{block_split}" --report "{block_report}" --chunks-per-block {n}


In [ ]:
for n in [2, 3, 4]:
    block_split = ROOT / f'data/literary/meta/literary_source_heldout_blocks_{n}.parquet'
    out_dir = ROOT / f'artifacts/literary_style_embedding_styledistance_blocks_{n}'
    !python scripts/style_embedding_recall.py --input "{block_split}" --out-dir "{out_dir}" --model-name StyleDistance/styledistance --batch-size 128


In [ ]:
import json
import pandas as pd

rows = []
for n in [2, 3, 4]:
    out_dir = ROOT / f'artifacts/literary_style_embedding_styledistance_blocks_{n}'
    metrics = json.loads((out_dir / 'style_embedding_metrics.json').read_text())
    metrics['chunks_per_block'] = n
    rows.append(metrics)
pd.DataFrame(rows)[['chunks_per_block', 'n_train', 'n_eval', 'n_authors', 'top1_accuracy', 'top3_accuracy', 'top5_accuracy', 'mrr']]
